# Simulation Log Analysis Report

Use this notebook to sanity-check a simulation run. Update `LOGS_DIR` below if your logs live elsewhere.


## 0. Setup

This section imports the required libraries (`networkx`, `numpy`) and identifies the log files to analyse.


In [ ]:
import json
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages
from tabulate import tabulate

import networkx as nx
import numpy as np

# LOGS_DIR = Path("results/20251024/baseline-honest-no-drop-small-degree-20251024-075646")
ALLOWED_SUFFIXES = {".log", ".jsonl", ".txt"}

def parse_timestamp(value: Optional[str]) -> Optional[datetime]:
    if not value or not isinstance(value, str):
        return None
    try:
        if value.endswith("Z"):
            value = value[:-1] + "+00:00"
        return datetime.fromisoformat(value)
    except ValueError:
        return None

def find_log_files(root: Path, allowed_suffixes: Iterable[str]) -> List[Path]:
    root = root.expanduser().resolve()
    if not root.exists():
        print(f"[warn] log directory {root} does not exist.")
        return []
    suffixes = {suffix.lower() for suffix in allowed_suffixes}
    files = sorted(
        p for p in root.rglob("*")
        if p.is_file() and (not suffixes or p.suffix.lower() in suffixes)
    )
    if not files:
        print(f"[warn] no log files with suffixes {sorted(suffixes)} found under {root}.")
    return files

def percentile(values: Sequence[float], pct: float) -> Optional[float]:
    arr = np.asarray(values, dtype=float)
    if arr.size == 0:
        return None
    return float(np.percentile(arr, pct))

# LOG_FILES = find_log_files(LOGS_DIR, ALLOWED_SUFFIXES)
# print(f"Discovered {len(LOG_FILES)} candidate log files under {LOGS_DIR.resolve()}")


In [2]:
def load_structured_logs(paths: Sequence[Path]) -> List[Dict]:
    records: List[Dict] = []
    skipped = 0
    for path in paths:
        with path.open("r", encoding="utf-8") as handle:
            for line_no, raw in enumerate(handle, 1):
                line = raw.strip()
                if not line:
                    continue
                try:
                    payload = json.loads(line)
                except json.JSONDecodeError:
                    skipped += 1
                    continue
                payload["_file"] = str(path)
                payload["_line"] = line_no
                payload["_timestamp"] = parse_timestamp(payload.get("timestamp") or payload.get("ts"))
                records.append(payload)
    print(f"Loaded {len(records)} structured log lines (skipped {skipped} non-JSON lines)")
    return records

# records = load_structured_logs(LOG_FILES)


## 1. Network Graph Checks

Build an undirected graph from neighbour events and report degree statistics and connected components.


In [3]:
def build_network_graph(records: Sequence[Dict]) -> nx.Graph:
    graph = nx.Graph()
    for rec in records:
        node_id = rec.get("node_id")
        if isinstance(node_id, str):
            graph.add_node(node_id)
        msg = rec.get("msg")
        if msg == "Final neighbor list":
            neighbors = rec.get("neighbors") or []
            for neighbor in neighbors:
                if isinstance(neighbor, str) and node_id and neighbor and node_id != neighbor:
                    graph.add_edge(node_id, neighbor)
        elif msg == "Successfully added neighbor":
            peer_id = rec.get("peer_id")
            if isinstance(node_id, str) and isinstance(peer_id, str) and node_id != peer_id:
                graph.add_edge(node_id, peer_id)
    return graph

def calculate_degree_statistics(graph: nx.Graph) -> Dict:
    if not graph.number_of_nodes():
        return {}
    
    degree_values = np.asarray([deg for _, deg in graph.degree()], dtype=int)
    if not degree_values.size:
        return {}
    
    return {
        "min": int(degree_values.min()),
        "max": int(degree_values.max()),
        "mean": float(np.mean(degree_values)),
        "median": float(np.median(degree_values)),
        "p95": percentile(degree_values, 95),
    }

def get_top_nodes_by_degree(graph: nx.Graph, top_n: int = 10) -> List[Tuple[str, int]]:
    return sorted(graph.degree, key=lambda item: item[1], reverse=True)[:top_n]

def analyze_connected_components(graph: nx.Graph) -> Dict:
    components = list(nx.connected_components(graph))
    if not components:
        return {}
    
    sizes = np.asarray([len(component) for component in components], dtype=int)
    sizes_sorted = np.sort(sizes)[::-1]
    
    return {
        "count": len(components),
        "largest": int(sizes_sorted[0]),
        "smallest": int(sizes_sorted[-1]),
        "sizes_sorted": sizes_sorted,
    }

def print_network_summary(graph: nx.Graph, top_nodes_count: int = 10):
    print(f"Nodes discovered: {graph.number_of_nodes()}")
    print(f"Inferred undirected edges: {graph.number_of_edges()}")
    
    if not graph.number_of_nodes():
        print("No node information was found in the logs.")
        return
    
    degree_stats = calculate_degree_statistics(graph)
    if degree_stats:
        print("Degree summary:", degree_stats)
        
        top_nodes = get_top_nodes_by_degree(graph, top_nodes_count)
        if top_nodes:
            print("Top nodes by degree:")
            for node, degree in top_nodes:
                print(f"  {node}: {degree}")
    
    component_stats = analyze_connected_components(graph)
    if component_stats:
        print(f"Connected components: {component_stats['count']}")
        print(f"Largest component size: {component_stats['largest']}")
        print(f"Smallest component size: {component_stats['smallest']}")
        if component_stats['count'] > 1:
            print("Component sizes (top 10):", ", ".join(str(int(s)) for s in component_stats['sizes_sorted'][:10]))

# graph = build_network_graph(records)
# print_network_summary(graph)


## 2. Protocol Message Statistics

Summarise per-protocol message volumes from `Message counts` log entries.


In [4]:
def extract_message_counts(records: Sequence[Dict]) -> Dict[str, List[Dict]]:
    buckets: Dict[str, List[Dict]] = defaultdict(list)
    for rec in records:
        if rec.get("msg") != "Message counts":
            continue
        protocol = rec.get("logger", "unknown")
        node_id = rec.get("node_id", "unknown")
        totals = rec.get("total_messages") or rec.get("totalMessages") or []
        valids = rec.get("valid_messages") or rec.get("validMessages") or []
        if not isinstance(totals, list):
            continue
        totals_arr = np.asarray(totals, dtype=int)
        valids_arr = np.asarray(valids, dtype=int) if isinstance(valids, list) else np.array([], dtype=int)
        buckets[protocol].append(
            {
                "node": node_id,
                "totals": totals_arr,
                "valids": valids_arr,
                "file": rec.get("_file"),
            }
        )
    return buckets

def flatten(arrays: Iterable[np.ndarray]) -> np.ndarray:
    arrays = list(arrays)
    if not arrays:
        return np.array([], dtype=float)
    return np.concatenate([np.asarray(a, dtype=float) for a in arrays if a.size])

def build_message_statistics_table(message_counts: Dict[str, List[Dict]]) -> pd.DataFrame:
    table_data = []
    
    for protocol, entries in sorted(message_counts.items()):
        totals = flatten(entry["totals"] for entry in entries)
        valids = flatten(entry["valids"] for entry in entries)
        totals_per_node = np.asarray([entry["totals"].sum() for entry in entries], dtype=float)
        
        row = {
            "Protocol": protocol,
            "Nodes": len(entries),
            "Total Rounds": int(totals.size),
            "Avg Msg/Round": totals.mean() if totals.size else 0,
            "P95 Msg/Round": percentile(totals, 95) if totals.size else 0,
            "Avg Msg/Node": totals_per_node.mean() if totals_per_node.size else 0,
            "Min Msg/Node": totals_per_node.min() if totals_per_node.size else 0,
            "Max Msg/Node": totals_per_node.max() if totals_per_node.size else 0,
            "Valid Ratio": float(valids.sum() / totals.sum()) if totals.size and valids.size and totals.sum() else float("nan")
        }
        table_data.append(row)
    
    return pd.DataFrame(table_data)

def create_message_statistics_plots(df: pd.DataFrame):
    if df.empty:
        return None
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle('Protocol Message Statistics', fontsize=16, fontweight='bold')
    
    protocols = df["Protocol"].tolist()
    
    ax1 = axes[0, 0]
    ax1.bar(protocols, df["Avg Msg/Round"], color='steelblue', alpha=0.7)
    ax1.set_ylabel('Messages')
    ax1.set_title('Average Messages per Round')
    ax1.tick_params(axis='x', rotation=45)
    ax1.grid(axis='y', alpha=0.3)
    
    ax2 = axes[0, 1]
    ax2.bar(protocols, df["P95 Msg/Round"], color='coral', alpha=0.7)
    ax2.set_ylabel('Messages')
    ax2.set_title('95th Percentile Messages per Round')
    ax2.tick_params(axis='x', rotation=45)
    ax2.grid(axis='y', alpha=0.3)
    
    ax3 = axes[1, 0]
    x = np.arange(len(protocols))
    width = 0.25
    ax3.bar(x - width, df["Min Msg/Node"], width, label='Min', color='lightgreen', alpha=0.7)
    ax3.bar(x, df["Avg Msg/Node"], width, label='Avg', color='green', alpha=0.7)
    ax3.bar(x + width, df["Max Msg/Node"], width, label='Max', color='darkgreen', alpha=0.7)
    ax3.set_ylabel('Total Messages')
    ax3.set_title('Messages per Node (Min/Avg/Max)')
    ax3.set_xticks(x)
    ax3.set_xticklabels(protocols, rotation=45)
    ax3.legend()
    ax3.grid(axis='y', alpha=0.3)
    
    ax4 = axes[1, 1]
    valid_ratios = [r if not np.isnan(r) else 0 for r in df["Valid Ratio"]]
    bars = ax4.bar(protocols, valid_ratios, color='mediumpurple', alpha=0.7)
    ax4.set_ylabel('Ratio')
    ax4.set_title('Valid Message Ratio')
    ax4.set_ylim(0, 1.1)
    ax4.tick_params(axis='x', rotation=45)
    ax4.grid(axis='y', alpha=0.3)
    
    for bar in bars:
        height = bar.get_height()
        if height > 0:
            ax4.text(bar.get_x() + bar.get_width()/2., height,
                    f'{height:.2f}',
                    ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    return fig

# message_counts = extract_message_counts(records)

# if not message_counts:
#     print("No 'Message counts' entries were found in the logs.")
# else:
#     df = build_message_statistics_table(message_counts)
#     print(tabulate(df, headers='keys', tablefmt='grid', showindex=False))
#     create_message_statistics_plots(df)
#     plt.show()


## 3. Committee Analysis

Analyze committee selection consistency and similarity across nodes.


In [5]:
from itertools import combinations
from scipy import stats

def extract_committees_by_node(records: Sequence[Dict]) -> Dict[str, List[str]]:
    node_committees = {}
    for rec in records:
        if rec.get("msg") == "Committee elected":
            node_id = rec.get("node_id")
            committee = rec.get("committee") or []
            if not node_id or not isinstance(committee, list):
                continue
            
            member_ids = []
            for member in committee:
                member_id = None
                if isinstance(member, dict):
                    member_id = member.get("ID") or member.get("id")
                elif isinstance(member, (list, tuple)) and member:
                    member_id = member[0]
                elif isinstance(member, str):
                    member_id = member
                
                if member_id:
                    member_ids.append(str(member_id))
            
            node_committees[str(node_id)] = member_ids
    
    return node_committees

def compute_committee_similarity(committees: Dict[str, List[str]]) -> Tuple[pd.DataFrame, Dict]:
    if not committees:
        return pd.DataFrame(), {}
    
    committee_sets = {node: set(members) for node, members in committees.items()}
    node_ids = sorted(committee_sets.keys())
    
    if committee_sets:
        global_intersection = set.intersection(*committee_sets.values())
    else:
        global_intersection = set()
    
    first_committee = next(iter(committee_sets.values()))
    full_consensus = all(comm == first_committee for comm in committee_sets.values())
    
    pairwise_data = []
    for node_a, node_b in combinations(node_ids, 2):
        set_a = committee_sets[node_a]
        set_b = committee_sets[node_b]
        
        intersection = set_a & set_b
        union = set_a | set_b
        
        intersection_size = len(intersection)
        union_size = len(union)
        size_a = len(set_a)
        size_b = len(set_b)
        
        jaccard = intersection_size / union_size if union_size > 0 else 0.0
        dice = (2 * intersection_size) / (size_a + size_b) if (size_a + size_b) > 0 else 0.0
        overlap = intersection_size / min(size_a, size_b) if min(size_a, size_b) > 0 else 0.0
        
        pairwise_data.append({
            "Node A": node_a,
            "Node B": node_b,
            "Size A": size_a,
            "Size B": size_b,
            "Intersection": intersection_size,
            "Union": union_size,
            "Jaccard": jaccard,
            "Dice": dice,
            "Overlap": overlap,
            "Common %": (intersection_size / size_a * 100) if size_a > 0 else 0.0,
        })
    
    pairwise_df = pd.DataFrame(pairwise_data)
    
    all_members = set()
    for members in committee_sets.values():
        all_members.update(members)
    all_members = sorted(all_members)
    
    membership_matrix = np.zeros((len(node_ids), len(all_members)), dtype=int)
    for i, node in enumerate(node_ids):
        for j, member in enumerate(all_members):
            if member in committee_sets[node]:
                membership_matrix[i, j] = 1
    
    correlations = []
    if len(node_ids) >= 2 and len(all_members) > 0:
        for node_a, node_b in combinations(node_ids, 2):
            idx_a = node_ids.index(node_a)
            idx_b = node_ids.index(node_b)
            vec_a = membership_matrix[idx_a]
            vec_b = membership_matrix[idx_b]
            
            if vec_a.std() > 0 and vec_b.std() > 0:
                corr, _ = stats.pearsonr(vec_a, vec_b)
                correlations.append({
                    "Node A": node_a,
                    "Node B": node_b,
                    "Pearson Corr": corr
                })
    
    correlation_df = pd.DataFrame(correlations) if correlations else pd.DataFrame()
    
    kappa_scores = []
    if len(node_ids) >= 2:
        for node_a, node_b in combinations(node_ids, 2):
            idx_a = node_ids.index(node_a)
            idx_b = node_ids.index(node_b)
            vec_a = membership_matrix[idx_a]
            vec_b = membership_matrix[idx_b]
            
            try:
                p_o = np.mean(vec_a == vec_b)
                p_yes_a = np.mean(vec_a)
                p_no_a = 1 - p_yes_a
                p_yes_b = np.mean(vec_b)
                p_no_b = 1 - p_yes_b
                p_e = p_yes_a * p_yes_b + p_no_a * p_no_b
                
                kappa = (p_o - p_e) / (1 - p_e) if (1 - p_e) > 0 else 0.0
                kappa_scores.append({
                    "Node A": node_a,
                    "Node B": node_b,
                    "Cohen's Kappa": kappa
                })
            except:
                pass
    
    kappa_df = pd.DataFrame(kappa_scores) if kappa_scores else pd.DataFrame()
    
    if not correlation_df.empty:
        pairwise_df = pairwise_df.merge(correlation_df, on=["Node A", "Node B"], how="left")
    if not kappa_df.empty:
        pairwise_df = pairwise_df.merge(kappa_df, on=["Node A", "Node B"], how="left")
    
    summary = {
        "total_nodes": len(node_ids),
        "total_unique_members": len(all_members),
        "global_intersection_size": len(global_intersection),
        "global_intersection_members": sorted(global_intersection),
        "full_consensus": full_consensus,
        "avg_committee_size": np.mean([len(c) for c in committee_sets.values()]),
        "min_committee_size": min(len(c) for c in committee_sets.values()) if committee_sets else 0,
        "max_committee_size": max(len(c) for c in committee_sets.values()) if committee_sets else 0,
    }
    
    if not pairwise_df.empty:
        summary["avg_jaccard"] = pairwise_df["Jaccard"].mean()
        summary["min_jaccard"] = pairwise_df["Jaccard"].min()
        summary["max_jaccard"] = pairwise_df["Jaccard"].max()
        summary["avg_dice"] = pairwise_df["Dice"].mean()
        summary["avg_overlap"] = pairwise_df["Overlap"].mean()
        if "Pearson Corr" in pairwise_df.columns:
            summary["avg_pearson_corr"] = pairwise_df["Pearson Corr"].mean()
        if "Cohen's Kappa" in pairwise_df.columns:
            summary["avg_cohens_kappa"] = pairwise_df["Cohen's Kappa"].mean()
    
    return pairwise_df, summary

def analyze_committee_selection(records: Sequence[Dict]) -> Tuple[Counter, List[int]]:
    committee_logs = [rec for rec in records if rec.get("msg") == "Committee elected"]
    
    selection_counter = Counter()
    committee_sizes = []
    for rec in committee_logs:
        committee = rec.get("committee") or []
        if isinstance(committee, list):
            committee_sizes.append(len(committee))
            for member in committee:
                member_id = None
                if isinstance(member, dict):
                    member_id = member.get("ID") or member.get("id")
                elif isinstance(member, (list, tuple)) and member:
                    member_id = member[0]
                if member_id:
                    selection_counter[str(member_id)] += 1
    
    return selection_counter, committee_sizes

def print_committee_summary(records: Sequence[Dict]):
    committee_logs = [rec for rec in records if rec.get("msg") == "Committee elected"]
    
    if not committee_logs:
        print("No committee election logs found.")
        return
    
    selection_counter, committee_sizes = analyze_committee_selection(records)
    
    print(f"Committee elections observed: {len(committee_logs)}")
    if committee_sizes:
        sizes = np.asarray(committee_sizes, dtype=float)
        print(f"  Average committee size: {sizes.mean():.2f}")
        print(f"  Committee size range: {int(sizes.min())} - {int(sizes.max())}")
    if selection_counter:
        counts = np.asarray(list(selection_counter.values()), dtype=float)
        print(f"  Unique members selected: {len(selection_counter)}")
        print(f"  Min selections per member: {int(counts.min())}")
        print(f"  Max selections per member: {int(counts.max())}")
        p95 = percentile(counts, 95)
        if p95 is not None:
            print(f"  95th percentile selections: {p95:.2f}")
        print("  Top-selected members:")
        for member_id, count in selection_counter.most_common(10):
            print(f"    {member_id}: {count}")
    else:
        print("  Committee membership data was not available in the logs.")
    
    node_committees = extract_committees_by_node(records)
    
    if not node_committees:
        print("\nNo committee data available for similarity comparison.")
    elif len(node_committees) < 2:
        print(f"\nOnly {len(node_committees)} node(s) have committee data. Need at least 2 for comparison.")
    else:
        print(f"\nAnalyzing committee similarity across {len(node_committees)} nodes...")
        
        pairwise_df, summary = compute_committee_similarity(node_committees)
        
        print(f"Total nodes: {summary['total_nodes']}")
        print(f"Total unique members across all committees: {summary['total_unique_members']}")
        print(f"Average committee size: {summary['avg_committee_size']:.2f}")
        print(f"Committee size range: {int(summary['min_committee_size'])} - {int(summary['max_committee_size'])}")
        print(f"Global intersection (members ALL nodes agreed on): {summary['global_intersection_size']}")
        if summary['global_intersection_members'] and len(summary['global_intersection_members']) <= 10:
            print(f"  Members: {summary['global_intersection_members']}")
        print(f"Full consensus (all committees identical): {summary['full_consensus']}")
        
        if not pairwise_df.empty:
            print(f"\nPairwise similarity metrics (across {len(pairwise_df)} node pairs):")
            print(f"  Average Jaccard index: {summary['avg_jaccard']:.4f}")
            print(f"  Jaccard range: {summary['min_jaccard']:.4f} - {summary['max_jaccard']:.4f}")
            print(f"  Average Dice coefficient: {summary['avg_dice']:.4f}")
            print(f"  Average Overlap coefficient: {summary['avg_overlap']:.4f}")
            if 'avg_pearson_corr' in summary:
                print(f"  Average Pearson correlation: {summary['avg_pearson_corr']:.4f}")
            if 'avg_cohens_kappa' in summary:
                print(f"  Average Cohen's kappa: {summary['avg_cohens_kappa']:.4f}")

# print_committee_summary(records)


## 4. Error & Warning Audit

Surface warning and error log entries to highlight potential issues that may need manual inspection.


In [6]:
def summarize_alerts(records: Sequence[Dict]) -> List[Dict]:
    alerts: List[Dict] = []
    for rec in records:
        level = str(rec.get("level", "")).upper()
        if level in {"ERROR", "WARN", "WARNING"}:
            alerts.append(rec)
    return alerts

def count_alerts_by_node(alerts: List[Dict]) -> Counter:
    by_node = Counter()
    for entry in alerts:
        node = entry.get("node_id", "global")
        level = str(entry.get("level", "")).upper()
        if level == "WARNING":
            level = "WARN"
        by_node[(node, level)] += 1
    return by_node

def print_alert_summary(records: Sequence[Dict]):
    alerts = summarize_alerts(records)
    if not alerts:
        print("No warnings or errors detected in the logs.")
        return
    
    by_node = count_alerts_by_node(alerts)
    print("Warnings/errors by node:")
    for (node, level), count in sorted(by_node.items(), key=lambda kv: (-kv[1], kv[0][0], kv[0][1])):
        print(f"  {level:<5} | {node}: {count}")
    print("Recent alert examples:")
    for entry in alerts[:5]:
        ts = entry.get("_timestamp") or entry.get("timestamp")
        ts_str = ts.isoformat() if isinstance(ts, datetime) else str(ts)
        print(f"  [{entry.get('level')}] {ts_str} {entry.get('node_id', 'global')}: {entry.get('msg')}")

# print_alert_summary(records)


## Batch Analysis: Process Multiple Simulations

Run the complete analysis pipeline on multiple simulation directories.

In [7]:
def extract_simulation_analysis(sim_path: Path, sim_name: str) -> Dict:
    log_files = find_log_files(sim_path, ALLOWED_SUFFIXES)
    
    if not log_files:
        return None
    
    sim_records = load_structured_logs(log_files)
    
    if not sim_records:
        return None
    
    sim_graph = build_network_graph(sim_records)
    sim_message_counts = extract_message_counts(sim_records)
    
    analysis = {
        "name": sim_name,
        "num_files": len(log_files),
        "num_records": len(sim_records),
        "graph": sim_graph,
        "message_counts": sim_message_counts,
        "records": sim_records,
    }
    
    degree_stats = calculate_degree_statistics(sim_graph)
    component_stats = analyze_connected_components(sim_graph)
    
    analysis["degree_stats"] = degree_stats
    analysis["component_stats"] = component_stats
    
    if sim_message_counts:
        analysis["message_stats_df"] = build_message_statistics_table(sim_message_counts)
        analysis["per_protocol_dfs"] = build_per_round_statistics_by_protocol(sim_message_counts)
        analysis["correlations"] = compute_degree_message_correlation(sim_graph, sim_message_counts)
    
    selection_counter, committee_sizes = analyze_committee_selection(sim_records)
    analysis["selection_counter"] = selection_counter
    analysis["committee_sizes"] = committee_sizes
    
    node_committees = extract_committees_by_node(sim_records)
    if node_committees and len(node_committees) >= 2:
        pairwise_df, summary = compute_committee_similarity(node_committees)
        analysis["committee_pairwise_df"] = pairwise_df
        analysis["committee_summary"] = summary
    
    alerts = summarize_alerts(sim_records)
    analysis["alerts"] = alerts
    if alerts:
        analysis["alerts_by_node"] = count_alerts_by_node(alerts)
    
    return analysis

def write_markdown_report(analysis: Dict, md_file):
    md_file.write(f"# Simulation Analysis: {analysis['name']}\n\n")
    md_file.write("---\n\n")
    
    md_file.write(f"**Files:** {analysis['num_files']} | **Records:** {analysis['num_records']}\n\n")
    
    md_file.write("## Network Graph Analysis\n\n")
    graph = analysis["graph"]
    md_file.write(f"- **Nodes:** {graph.number_of_nodes()}\n")
    md_file.write(f"- **Edges:** {graph.number_of_edges()}\n")
    
    if analysis.get("degree_stats"):
        deg = analysis["degree_stats"]
        md_file.write(f"- **Degree:** min={deg['min']}, max={deg['max']}, ")
        md_file.write(f"mean={deg['mean']:.1f}, median={deg['median']:.1f}\n")
    
    if analysis.get("component_stats"):
        comp = analysis["component_stats"]
        md_file.write(f"- **Components:** {comp['count']}, Largest: {comp['largest']}\n")
    
    md_file.write("\n")
    
    if analysis.get("message_stats_df") is not None and not analysis["message_stats_df"].empty:
        md_file.write("## Protocol Message Statistics\n\n")
        df = analysis["message_stats_df"]
        md_file.write(tabulate(df, headers='keys', tablefmt='pipe', showindex=False))
        md_file.write("\n\n")
    
    if analysis.get("per_protocol_dfs"):
        md_file.write("## Per-Round Statistics\n\n")
        for protocol, df in analysis["per_protocol_dfs"].items():
            if not df.empty:
                md_file.write(f"### {protocol}\n\n")
                md_file.write(tabulate(df.head(10), headers='keys', tablefmt='pipe', showindex=False, floatfmt='.1f'))
                md_file.write("\n\n")
    
    if analysis.get("correlations"):
        md_file.write("## Degree-Message Correlation\n\n")
        md_file.write("*Pearson correlation between node degree and total messages*\n\n")
        
        corr_data = []
        for protocol, data in sorted(analysis["correlations"].items()):
            corr_data.append({
                "Protocol": protocol,
                "Correlation": f"{data['correlation']:.4f}",
                "P-value": f"{data['p_value']:.4f}",
                "Sample": data['sample_size'],
                "Avg Degree": f"{data['avg_degree']:.1f}",
                "Avg Msgs": f"{data['avg_messages']:.1f}"
            })
        
        corr_df = pd.DataFrame(corr_data)
        md_file.write(tabulate(corr_df, headers='keys', tablefmt='pipe', showindex=False))
        md_file.write("\n\n")
    
    md_file.write("## Committee Analysis\n\n")
    
    if analysis.get("committee_sizes"):
        sizes = np.asarray(analysis["committee_sizes"], dtype=float)
        elections = len([r for r in analysis["records"] if r.get('msg') == 'Committee elected'])
        md_file.write(f"- **Elections:** {elections}\n")
        md_file.write(f"- **Avg committee size:** {sizes.mean():.1f}\n")
        md_file.write(f"- **Size range:** {int(sizes.min())}-{int(sizes.max())}\n")
        
        if analysis.get("selection_counter"):
            counts = np.asarray(list(analysis["selection_counter"].values()), dtype=float)
            md_file.write(f"- **Unique members:** {len(analysis['selection_counter'])}\n")
            md_file.write(f"- **Selections per member:** {int(counts.min())}-{int(counts.max())}\n")
    else:
        md_file.write("No committee data available.\n")
    
    md_file.write("\n")
    
    if analysis.get("committee_summary"):
        summary = analysis["committee_summary"]
        md_file.write("### Committee Similarity\n\n")
        md_file.write(f"- **Nodes compared:** {summary['total_nodes']}\n")
        md_file.write(f"- **Unique members:** {summary['total_unique_members']}\n")
        md_file.write(f"- **Global intersection:** {summary['global_intersection_size']}\n")
        md_file.write(f"- **Full consensus:** {summary['full_consensus']}\n")
        
        if not analysis.get("committee_pairwise_df", pd.DataFrame()).empty:
            md_file.write(f"- **Avg Jaccard:** {summary['avg_jaccard']:.3f}\n")
            md_file.write(f"- **Avg Dice:** {summary['avg_dice']:.3f}\n")
            md_file.write(f"- **Avg Overlap:** {summary['avg_overlap']:.3f}\n")
        
        md_file.write("\n")
    
    md_file.write("## Error & Warning Audit\n\n")
    
    if analysis.get("alerts"):
        md_file.write(f"- **Total alerts:** {len(analysis['alerts'])}\n")
        
        if analysis.get("alerts_by_node"):
            md_file.write("\n### Top Alerts\n\n")
            top_alerts = sorted(analysis["alerts_by_node"].items(), key=lambda kv: -kv[1])[:5]
            for (node, level), count in top_alerts:
                md_file.write(f"- **{level}** | {node[:50]}: {count}\n")
    else:
        md_file.write("No alerts detected.\n")
    
    md_file.write("\n---\n\n")


In [8]:
def process_simulation_for_pdf_from_analysis(analysis: Dict, pdf: PdfPages):
    if not analysis:
        return
    
    sim_name = analysis["name"]
    sim_graph = analysis["graph"]
    sim_message_counts = analysis["message_counts"]
    sim_records = analysis["records"]
    
    fig = plt.figure(figsize=(11, 17))
    fig.suptitle(f'Simulation Analysis: {sim_name}', fontsize=14, fontweight='bold', y=0.99)
    
    text_content = []
    text_content.append(f"SIMULATION: {sim_name}\n")
    text_content.append("="*80 + "\n\n")
    
    text_content.append(f"Files: {analysis['num_files']}, Records: {analysis['num_records']}\n\n")
    
    text_content.append("Network Graph Analysis:\n")
    text_content.append(f"  Nodes: {sim_graph.number_of_nodes()}, Edges: {sim_graph.number_of_edges()}\n")
    
    degree_stats = analysis.get("degree_stats")
    if degree_stats:
        text_content.append(f"  Degree: min={degree_stats['min']}, max={degree_stats['max']}, ")
        text_content.append(f"mean={degree_stats['mean']:.1f}, median={degree_stats['median']:.1f}\n")
    
    component_stats = analysis.get("component_stats")
    if component_stats:
        text_content.append(f"  Components: {component_stats['count']}, ")
        text_content.append(f"Largest: {component_stats['largest']}\n\n")
    else:
        text_content.append("\n")
    
    ax_text = fig.add_subplot(6, 1, 1)
    ax_text.axis('off')
    ax_text.text(0.05, 0.95, ''.join(text_content), transform=ax_text.transAxes,
                 fontsize=9, verticalalignment='top', fontfamily='monospace')
    
    ax_table = fig.add_subplot(6, 1, 2)
    ax_table.axis('off')
    
    if analysis.get("message_stats_df") is not None:
        df_msg = analysis["message_stats_df"]
        ax_table.text(0.05, 0.95, "Protocol Message Statistics:",
                        transform=ax_table.transAxes, fontsize=10, fontweight='bold',
                        verticalalignment='top')
        
        table_str = tabulate(df_msg, headers='keys', tablefmt='simple', showindex=False)
        ax_table.text(0.05, 0.85, table_str, transform=ax_table.transAxes,
                        fontsize=8, verticalalignment='top', fontfamily='monospace')
        
    ax_plots = fig.add_subplot(6, 1, 3)
    ax_plots.axis('off')
    
    if analysis.get("message_stats_df") is not None:
        df_msg = analysis["message_stats_df"]
        mini_fig, mini_axes = plt.subplots(2, 2, figsize=(10, 6))
        
        protocols = df_msg["Protocol"].tolist()
        
        mini_axes[0, 0].bar(protocols, df_msg["Avg Msg/Round"], color='steelblue', alpha=0.7)
        mini_axes[0, 0].set_ylabel('Messages', fontsize=8)
        mini_axes[0, 0].set_title('Avg Msg/Round', fontsize=8)
        mini_axes[0, 0].tick_params(axis='both', labelsize=7)
        mini_axes[0, 0].tick_params(axis='x', rotation=45)
        
        mini_axes[0, 1].bar(protocols, df_msg["P95 Msg/Round"], color='coral', alpha=0.7)
        mini_axes[0, 1].set_ylabel('Messages', fontsize=8)
        mini_axes[0, 1].set_title('P95 Msg/Round', fontsize=8)
        mini_axes[0, 1].tick_params(axis='both', labelsize=7)
        mini_axes[0, 1].tick_params(axis='x', rotation=45)
        
        x = np.arange(len(protocols))
        width = 0.25
        mini_axes[1, 0].bar(x - width, df_msg["Min Msg/Node"], width, label='Min', color='lightgreen', alpha=0.7)
        mini_axes[1, 0].bar(x, df_msg["Avg Msg/Node"], width, label='Avg', color='green', alpha=0.7)
        mini_axes[1, 0].bar(x + width, df_msg["Max Msg/Node"], width, label='Max', color='darkgreen', alpha=0.7)
        mini_axes[1, 0].set_ylabel('Messages', fontsize=8)
        mini_axes[1, 0].set_title('Msg/Node', fontsize=8)
        mini_axes[1, 0].set_xticks(x)
        mini_axes[1, 0].set_xticklabels(protocols, rotation=45, fontsize=7)
        mini_axes[1, 0].legend(fontsize=7)
        mini_axes[1, 0].tick_params(axis='y', labelsize=7)
        
        valid_ratios = [r if not np.isnan(r) else 0 for r in df_msg["Valid Ratio"]]
        mini_axes[1, 1].bar(protocols, valid_ratios, color='mediumpurple', alpha=0.7)
        mini_axes[1, 1].set_ylabel('Ratio', fontsize=8)
        mini_axes[1, 1].set_title('Valid Ratio', fontsize=8)
        mini_axes[1, 1].set_ylim(0, 1.1)
        mini_axes[1, 1].tick_params(axis='both', labelsize=7)
        mini_axes[1, 1].tick_params(axis='x', rotation=45)
        
        mini_fig.tight_layout()
        
        from io import BytesIO
        buf = BytesIO()
        mini_fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
        buf.seek(0)
        
        from PIL import Image
        img = Image.open(buf)
        ax_plots.imshow(img)
        
        plt.close(mini_fig)
        buf.close()
    
    ax_per_round = fig.add_subplot(6, 1, 4)
    ax_per_round.axis('off')
    
    if analysis.get("per_protocol_dfs"):
        per_protocol_dfs = analysis["per_protocol_dfs"]
        y_pos = 0.95
        for protocol, df in per_protocol_dfs.items():
            if not df.empty:
                ax_per_round.text(0.05, y_pos, f"{protocol} - Per-Round Stats:",
                                 transform=ax_per_round.transAxes, fontsize=9, fontweight='bold',
                                 verticalalignment='top')
                y_pos -= 0.05
                
                per_round_str = tabulate(df.head(10), headers='keys', tablefmt='simple', showindex=False, floatfmt='.1f')
                ax_per_round.text(0.05, y_pos, per_round_str, transform=ax_per_round.transAxes,
                                 fontsize=6, verticalalignment='top', fontfamily='monospace')
                y_pos -= (0.15 + 0.01 * min(len(df), 10))
                
                if y_pos < 0.05:
                    break
    
    ax_correlation = fig.add_subplot(6, 1, 5)
    ax_correlation.axis('off')
    
    if analysis.get("correlations"):
        correlations = analysis["correlations"]
        ax_correlation.text(0.05, 0.95, "Degree-Message Correlation:",
                           transform=ax_correlation.transAxes, fontsize=10, fontweight='bold',
                           verticalalignment='top')
        ax_correlation.text(0.05, 0.90, "(Pearson correlation between node degree and total messages)",
                           transform=ax_correlation.transAxes, fontsize=8,
                           verticalalignment='top', style='italic')
        
        corr_data = []
        for protocol, data in sorted(correlations.items()):
            corr_data.append({
                "Protocol": protocol,
                "Correlation": f"{data['correlation']:.4f}",
                "P-value": f"{data['p_value']:.4f}",
                "Sample": data['sample_size'],
                "Avg Degree": f"{data['avg_degree']:.1f}",
                "Avg Msgs": f"{data['avg_messages']:.1f}"
            })
        
        corr_df = pd.DataFrame(corr_data)
        corr_table_str = tabulate(corr_df, headers='keys', tablefmt='simple', showindex=False)
        ax_correlation.text(0.05, 0.82, corr_table_str, transform=ax_correlation.transAxes,
                           fontsize=8, verticalalignment='top', fontfamily='monospace')
    
    ax_footer = fig.add_subplot(6, 1, 6)
    ax_footer.axis('off')
    
    committee_text = []
    committee_text.append("Committee Analysis:\n")
    
    if analysis.get("committee_sizes"):
        sizes = np.asarray(analysis["committee_sizes"], dtype=float)
        elections = len([r for r in sim_records if r.get('msg') == 'Committee elected'])
        committee_text.append(f"  Elections: {elections}, ")
        committee_text.append(f"Avg size: {sizes.mean():.1f}, ")
        committee_text.append(f"Range: {int(sizes.min())}-{int(sizes.max())}\n")
        
        if analysis.get("selection_counter"):
            counts = np.asarray(list(analysis["selection_counter"].values()), dtype=float)
            committee_text.append(f"  Unique members: {len(analysis['selection_counter'])}, ")
            committee_text.append(f"Selections: {int(counts.min())}-{int(counts.max())}\n")
    else:
        committee_text.append("  No committee data\n")
    
    if analysis.get("committee_summary"):
        summary = analysis["committee_summary"]
        committee_text.append("\nSimilarity Analysis:\n")
        committee_text.append(f"  Nodes compared: {summary['total_nodes']}, ")
        committee_text.append(f"Unique members: {summary['total_unique_members']}\n")
        committee_text.append(f"  Global intersection: {summary['global_intersection_size']}, ")
        committee_text.append(f"Full consensus: {summary['full_consensus']}\n")
        if analysis.get("committee_pairwise_df") is not None and not analysis["committee_pairwise_df"].empty:
            committee_text.append(f"  Avg Jaccard: {summary['avg_jaccard']:.3f}, ")
            committee_text.append(f"Dice: {summary['avg_dice']:.3f}, ")
            committee_text.append(f"Overlap: {summary['avg_overlap']:.3f}\n")
    
    committee_text.append("\nError & Warning Audit:\n")
    if analysis.get("alerts"):
        alerts = analysis["alerts"]
        alerts_by_node = analysis.get("alerts_by_node", {})
        committee_text.append(f"  Total alerts: {len(alerts)}\n")
        top_alerts = sorted(alerts_by_node.items(), key=lambda kv: -kv[1])[:3]
        for (node, level), count in top_alerts:
            committee_text.append(f"    {level}: {node[:30]}: {count}\n")
    else:
        committee_text.append("  No alerts\n")
    
    ax_footer.text(0.05, 0.95, ''.join(committee_text), transform=ax_footer.transAxes,
                   fontsize=8, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

def process_batch_reports(batch_dir: Path, output_pdf: Path, output_md: Path):
    batch_dir = batch_dir.expanduser().resolve()
    
    if not batch_dir.exists():
        print(f"[ERROR] Directory does not exist: {batch_dir}")
        return
    
    items_to_process = []
    temp_dirs = []
    
    for item in sorted(batch_dir.iterdir()):
        if item.is_dir():
            items_to_process.append((item, item.name))
        elif item.is_file() and item.name.endswith(".tar.gz"):
            try:
                print(f"Extracting {item.name}...")
                extracted_path = extract_tar_gz(item)
                temp_dirs.append(extracted_path.parent if extracted_path != extracted_path.parent else extracted_path)
                items_to_process.append((extracted_path, item.stem.replace(".tar", "")))
            except Exception as e:
                print(f"[ERROR] Failed to extract {item.name}: {e}")
    
    if not items_to_process:
        print(f"[ERROR] No simulation data found in {batch_dir}")
        return
    
    print(f"\nProcessing {len(items_to_process)} simulation(s)...")
    print(f"PDF output: {output_pdf}")
    print(f"MD output: {output_md}\n")
    
    all_analyses = []
    
    for idx, (sim_path, sim_name) in enumerate(items_to_process, 1):
        print(f"[{idx}/{len(items_to_process)}] Analyzing {sim_name}...")
        try:
            analysis = extract_simulation_analysis(sim_path, sim_name)
            if analysis:
                all_analyses.append(analysis)
        except Exception as e:
            print(f"  [ERROR] Failed: {e}")
    
    if not all_analyses:
        print("[ERROR] No analyses were successful")
        return
    
    print(f"\nGenerating PDF report...")
    try:
        with PdfPages(output_pdf) as pdf:
            for analysis in all_analyses:
                process_simulation_for_pdf_from_analysis(analysis, pdf)
        print(f"✓ PDF generated: {output_pdf}")
    except Exception as e:
        print(f"  [ERROR] PDF generation failed: {e}")
    
    print(f"\nGenerating Markdown report...")
    try:
        with open(output_md, 'w', encoding='utf-8') as md_file:
            md_file.write("# Simulation Batch Analysis Report\n\n")
            md_file.write(f"**Total Simulations:** {len(all_analyses)}\n\n")
            md_file.write("---\n\n")
            
            for analysis in all_analyses:
                write_markdown_report(analysis, md_file)
        
        print(f"✓ Markdown generated: {output_md}")
    except Exception as e:
        print(f"  [ERROR] Markdown generation failed: {e}")
    
    for temp_dir in temp_dirs:
        if temp_dir.exists():
            shutil.rmtree(temp_dir, ignore_errors=True)
    
    print(f"\n{'='*80}")
    print(f"Reports completed!")
    print(f"  PDF: {output_pdf} ({len(all_analyses)} pages)")
    print(f"  MD:  {output_md}")
    print(f"{'='*80}")


In [9]:
import tarfile
import shutil
import tempfile
from io import StringIO
import sys

def compute_degree_message_correlation(graph: nx.Graph, message_counts: Dict[str, List[Dict]]) -> Dict:
    if graph.number_of_nodes() == 0 or not message_counts:
        return {}
    
    degree_dict = dict(graph.degree())
    
    correlations = {}
    for protocol, entries in message_counts.items():
        degrees = []
        total_messages = []
        
        for entry in entries:
            node_id = entry["node"]
            if node_id in degree_dict:
                degrees.append(degree_dict[node_id])
                total_messages.append(int(entry["totals"].sum()))
        
        if len(degrees) >= 2:
            degrees_arr = np.array(degrees)
            messages_arr = np.array(total_messages)
            
            if degrees_arr.std() > 0 and messages_arr.std() > 0:
                from scipy.stats import pearsonr
                corr, p_value = pearsonr(degrees_arr, messages_arr)
                correlations[protocol] = {
                    "correlation": corr,
                    "p_value": p_value,
                    "sample_size": len(degrees),
                    "avg_degree": degrees_arr.mean(),
                    "avg_messages": messages_arr.mean()
                }
    
    return correlations

def build_per_round_statistics_by_protocol(message_counts: Dict[str, List[Dict]]) -> Dict[str, pd.DataFrame]:
    per_protocol_stats = {}
    
    for protocol, entries in sorted(message_counts.items()):
        if not entries:
            continue
        
        max_rounds = max(len(entry["totals"]) for entry in entries)
        if max_rounds == 0:
            continue
        
        round_stats = []
        for round_idx in range(max_rounds):
            round_messages = []
            for entry in entries:
                if round_idx < len(entry["totals"]):
                    round_messages.append(entry["totals"][round_idx])
            
            if round_messages:
                round_messages_arr = np.array(round_messages)
                round_stats.append({
                    "Round": round_idx,
                    "Min": int(round_messages_arr.min()),
                    "Max": int(round_messages_arr.max()),
                    "Mean": round_messages_arr.mean(),
                    "Median": np.median(round_messages_arr),
                    "Std": round_messages_arr.std(),
                    "P25": percentile(round_messages_arr, 25),
                    "P75": percentile(round_messages_arr, 75),
                    "P95": percentile(round_messages_arr, 95),
                })
        
        if round_stats:
            per_protocol_stats[protocol] = pd.DataFrame(round_stats)
    
    return per_protocol_stats

def process_simulation_for_pdf(sim_path: Path, sim_name: str, pdf: PdfPages):
    log_files = find_log_files(sim_path, ALLOWED_SUFFIXES)
    
    if not log_files:
        return
    
    sim_records = load_structured_logs(log_files)
    
    if not sim_records:
        return
    
    sim_graph = build_network_graph(sim_records)
    sim_message_counts = extract_message_counts(sim_records)
    
    fig = plt.figure(figsize=(11, 17))
    fig.suptitle(f'Simulation Analysis: {sim_name}', fontsize=14, fontweight='bold', y=0.99)
    
    text_content = []
    text_content.append(f"SIMULATION: {sim_name}\n")
    text_content.append("="*80 + "\n\n")
    
    text_content.append(f"Files: {len(log_files)}, Records: {len(sim_records)}\n\n")
    
    text_content.append("Network Graph Analysis:\n")
    text_content.append(f"  Nodes: {sim_graph.number_of_nodes()}, Edges: {sim_graph.number_of_edges()}\n")
    
    degree_stats = calculate_degree_statistics(sim_graph)
    if degree_stats:
        text_content.append(f"  Degree: min={degree_stats['min']}, max={degree_stats['max']}, ")
        text_content.append(f"mean={degree_stats['mean']:.1f}, median={degree_stats['median']:.1f}\n")
    
    component_stats = analyze_connected_components(sim_graph)
    if component_stats:
        text_content.append(f"  Components: {component_stats['count']}, ")
        text_content.append(f"Largest: {component_stats['largest']}\n\n")
    else:
        text_content.append("\n")
    
    ax_text = fig.add_subplot(6, 1, 1)
    ax_text.axis('off')
    ax_text.text(0.05, 0.95, ''.join(text_content), transform=ax_text.transAxes,
                 fontsize=9, verticalalignment='top', fontfamily='monospace')
    
    ax_table = fig.add_subplot(6, 1, 2)
    ax_table.axis('off')
    
    if sim_message_counts:
        df_msg = build_message_statistics_table(sim_message_counts)
        ax_table.text(0.05, 0.95, "Protocol Message Statistics:",
                        transform=ax_table.transAxes, fontsize=10, fontweight='bold',
                        verticalalignment='top')
        
        table_str = tabulate(df_msg, headers='keys', tablefmt='simple', showindex=False)
        ax_table.text(0.05, 0.85, table_str, transform=ax_table.transAxes,
                        fontsize=8, verticalalignment='top', fontfamily='monospace')
        
    ax_plots = fig.add_subplot(6, 1, 3)
    ax_plots.axis('off')
    
    mini_fig, mini_axes = plt.subplots(2, 2, figsize=(10, 6))
    
    protocols = df_msg["Protocol"].tolist()
    
    mini_axes[0, 0].bar(protocols, df_msg["Avg Msg/Round"], color='steelblue', alpha=0.7)
    mini_axes[0, 0].set_ylabel('Messages', fontsize=8)
    mini_axes[0, 0].set_title('Avg Msg/Round', fontsize=8)
    mini_axes[0, 0].tick_params(axis='both', labelsize=7)
    mini_axes[0, 0].tick_params(axis='x', rotation=45)
    
    mini_axes[0, 1].bar(protocols, df_msg["P95 Msg/Round"], color='coral', alpha=0.7)
    mini_axes[0, 1].set_ylabel('Messages', fontsize=8)
    mini_axes[0, 1].set_title('P95 Msg/Round', fontsize=8)
    mini_axes[0, 1].tick_params(axis='both', labelsize=7)
    mini_axes[0, 1].tick_params(axis='x', rotation=45)
    
    x = np.arange(len(protocols))
    width = 0.25
    mini_axes[1, 0].bar(x - width, df_msg["Min Msg/Node"], width, label='Min', color='lightgreen', alpha=0.7)
    mini_axes[1, 0].bar(x, df_msg["Avg Msg/Node"], width, label='Avg', color='green', alpha=0.7)
    mini_axes[1, 0].bar(x + width, df_msg["Max Msg/Node"], width, label='Max', color='darkgreen', alpha=0.7)
    mini_axes[1, 0].set_ylabel('Messages', fontsize=8)
    mini_axes[1, 0].set_title('Msg/Node', fontsize=8)
    mini_axes[1, 0].set_xticks(x)
    mini_axes[1, 0].set_xticklabels(protocols, rotation=45, fontsize=7)
    mini_axes[1, 0].legend(fontsize=7)
    mini_axes[1, 0].tick_params(axis='y', labelsize=7)
    
    valid_ratios = [r if not np.isnan(r) else 0 for r in df_msg["Valid Ratio"]]
    mini_axes[1, 1].bar(protocols, valid_ratios, color='mediumpurple', alpha=0.7)
    mini_axes[1, 1].set_ylabel('Ratio', fontsize=8)
    mini_axes[1, 1].set_title('Valid Ratio', fontsize=8)
    mini_axes[1, 1].set_ylim(0, 1.1)
    mini_axes[1, 1].tick_params(axis='both', labelsize=7)
    mini_axes[1, 1].tick_params(axis='x', rotation=45)
    
    mini_fig.tight_layout()
    
    from io import BytesIO
    buf = BytesIO()
    mini_fig.savefig(buf, format='png', dpi=100, bbox_inches='tight')
    buf.seek(0)
    
    from PIL import Image
    img = Image.open(buf)
    ax_plots.imshow(img)
    
    plt.close(mini_fig)
    buf.close()
    
    ax_per_round = fig.add_subplot(6, 1, 4)
    ax_per_round.axis('off')
    
    if sim_message_counts:
        per_protocol_dfs = build_per_round_statistics_by_protocol(sim_message_counts)
        if per_protocol_dfs:
            y_pos = 0.95
            for protocol, df in per_protocol_dfs.items():
                if not df.empty:
                    ax_per_round.text(0.05, y_pos, f"{protocol} - Per-Round Stats:",
                                     transform=ax_per_round.transAxes, fontsize=9, fontweight='bold',
                                     verticalalignment='top')
                    y_pos -= 0.05
                    
                    # Show first 10 rounds
                    per_round_str = tabulate(df.head(10), headers='keys', tablefmt='simple', showindex=False, floatfmt='.1f')
                    ax_per_round.text(0.05, y_pos, per_round_str, transform=ax_per_round.transAxes,
                                     fontsize=6, verticalalignment='top', fontfamily='monospace')
                    y_pos -= (0.15 + 0.01 * min(len(df), 10))
                    
                    if y_pos < 0.05:
                        break
    
    ax_correlation = fig.add_subplot(6, 1, 5)
    ax_correlation.axis('off')
    
    if sim_message_counts:
        correlations = compute_degree_message_correlation(sim_graph, sim_message_counts)
        if correlations:
            ax_correlation.text(0.05, 0.95, "Degree-Message Correlation:",
                               transform=ax_correlation.transAxes, fontsize=10, fontweight='bold',
                               verticalalignment='top')
            ax_correlation.text(0.05, 0.90, "(Pearson correlation between node degree and total messages)",
                               transform=ax_correlation.transAxes, fontsize=8,
                               verticalalignment='top', style='italic')
            
            corr_data = []
            for protocol, data in sorted(correlations.items()):
                corr_data.append({
                    "Protocol": protocol,
                    "Correlation": f"{data['correlation']:.4f}",
                    "P-value": f"{data['p_value']:.4f}",
                    "Sample": data['sample_size'],
                    "Avg Degree": f"{data['avg_degree']:.1f}",
                    "Avg Msgs": f"{data['avg_messages']:.1f}"
                })
            
            corr_df = pd.DataFrame(corr_data)
            corr_table_str = tabulate(corr_df, headers='keys', tablefmt='simple', showindex=False)
            ax_correlation.text(0.05, 0.82, corr_table_str, transform=ax_correlation.transAxes,
                               fontsize=8, verticalalignment='top', fontfamily='monospace')
    
    ax_footer = fig.add_subplot(6, 1, 6)
    ax_footer.axis('off')
    
    committee_text = []
    committee_text.append("Committee Analysis:\n")
    
    selection_counter, committee_sizes = analyze_committee_selection(sim_records)
    if committee_sizes:
        sizes = np.asarray(committee_sizes, dtype=float)
        committee_text.append(f"  Elections: {len([r for r in sim_records if r.get('msg') == 'Committee elected'])}, ")
        committee_text.append(f"Avg size: {sizes.mean():.1f}, ")
        committee_text.append(f"Range: {int(sizes.min())}-{int(sizes.max())}\n")
        
        if selection_counter:
            counts = np.asarray(list(selection_counter.values()), dtype=float)
            committee_text.append(f"  Unique members: {len(selection_counter)}, ")
            committee_text.append(f"Selections: {int(counts.min())}-{int(counts.max())}\n")
    else:
        committee_text.append("  No committee data\n")
    
    sim_node_committees = extract_committees_by_node(sim_records)
    if sim_node_committees and len(sim_node_committees) >= 2:
        pairwise_df, summary = compute_committee_similarity(sim_node_committees)
        committee_text.append("\nSimilarity Analysis:\n")
        committee_text.append(f"  Nodes compared: {summary['total_nodes']}, ")
        committee_text.append(f"Unique members: {summary['total_unique_members']}\n")
        committee_text.append(f"  Global intersection: {summary['global_intersection_size']}, ")
        committee_text.append(f"Full consensus: {summary['full_consensus']}\n")
        if not pairwise_df.empty:
            committee_text.append(f"  Avg Jaccard: {summary['avg_jaccard']:.3f}, ")
            committee_text.append(f"Dice: {summary['avg_dice']:.3f}, ")
            committee_text.append(f"Overlap: {summary['avg_overlap']:.3f}\n")
    
    committee_text.append("\nError & Warning Audit:\n")
    alerts = summarize_alerts(sim_records)
    if alerts:
        by_node = count_alerts_by_node(alerts)
        committee_text.append(f"  Total alerts: {len(alerts)}\n")
        top_alerts = sorted(by_node.items(), key=lambda kv: -kv[1])[:3]
        for (node, level), count in top_alerts:
            committee_text.append(f"    {level}: {node[:30]}: {count}\n")
    else:
        committee_text.append("  No alerts\n")
    
    ax_footer.text(0.05, 0.95, ''.join(committee_text), transform=ax_footer.transAxes,
                   fontsize=8, verticalalignment='top', fontfamily='monospace')
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

def extract_tar_gz(tar_path: Path) -> Path:
    temp_dir = Path(tempfile.mkdtemp(prefix="sim_extract_"))
    with tarfile.open(tar_path, "r:gz") as tar:
        tar.extractall(path=temp_dir)
    
    extracted_items = list(temp_dir.iterdir())
    if len(extracted_items) == 1 and extracted_items[0].is_dir():
        return extracted_items[0]
    return temp_dir

def process_batch_to_pdf(batch_dir: Path, output_pdf: Path):
    batch_dir = batch_dir.expanduser().resolve()
    
    if not batch_dir.exists():
        print(f"[ERROR] Directory does not exist: {batch_dir}")
        return
    
    items_to_process = []
    temp_dirs = []
    
    for item in sorted(batch_dir.iterdir()):
        if item.is_dir():
            items_to_process.append((item, item.name))
        elif item.is_file() and item.name.endswith(".tar.gz"):
            try:
                print(f"Extracting {item.name}...")
                extracted_path = extract_tar_gz(item)
                temp_dirs.append(extracted_path.parent if extracted_path != extracted_path.parent else extracted_path)
                items_to_process.append((extracted_path, item.stem.replace(".tar", "")))
            except Exception as e:
                print(f"[ERROR] Failed to extract {item.name}: {e}")
    
    if not items_to_process:
        print(f"[ERROR] No simulation data found in {batch_dir}")
        return
    
    print(f"\nGenerating PDF report for {len(items_to_process)} simulation(s)...")
    print(f"Output: {output_pdf}\n")
    
    try:
        with PdfPages(output_pdf) as pdf:
            for idx, (sim_path, sim_name) in enumerate(items_to_process, 1):
                print(f"[{idx}/{len(items_to_process)}] Processing {sim_name}...")
                try:
                    process_simulation_for_pdf(sim_path, sim_name, pdf)
                except Exception as e:
                    print(f"  [ERROR] Failed: {e}")
        
        print(f"\n{'='*80}")
        print(f"PDF report generated: {output_pdf}")
        print(f"Total pages: {len(items_to_process)}")
        print(f"{'='*80}")
    
    finally:
        for temp_dir in temp_dirs:
            if temp_dir.exists():
                shutil.rmtree(temp_dir, ignore_errors=True)

BATCH_DIR = Path("results")
OUTPUT_PDF = BATCH_DIR / "simulation_batch_report.pdf"
OUTPUT_MD = BATCH_DIR / "simulation_batch_report.md"
process_batch_reports(BATCH_DIR, OUTPUT_PDF, OUTPUT_MD)


Extracting baseline-honest-no-drop-large-degree-20251024-131216.tar.gz...


/var/folders/ld/rqglbq693g37vv4gt0w2w32c0000gp/T/ipykernel_14403/3085020029.py:293: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=temp_dir)


Extracting baseline-honest-no-drop-large-degree-20251121-123121.tar.gz...
Extracting baseline-honest-no-drop-medium-degree-20251024-103431.tar.gz...
Extracting baseline-honest-no-drop-medium-degree-20251121-095333.tar.gz...
Extracting baseline-honest-no-drop-small-degree-20251024-075646.tar.gz...
Extracting baseline-honest-no-drop-small-degree-20251121-071547.tar.gz...
Extracting big-committee-heavy-drop-large-degree-20251121-202530.tar.gz...
Extracting big-committee-heavy-drop-medium-degree-20251121-182646.tar.gz...
Extracting big-committee-heavy-drop-small-degree-20251121-162815.tar.gz...
Extracting big-committee-with-drop-large-degree-20251024-151036.tar.gz...
Extracting big-committee-with-drop-large-degree-20251121-142947.tar.gz...
Extracting big-committee-with-drop-medium-degree-20251024-123248.tar.gz...
Extracting big-committee-with-drop-medium-degree-20251121-115152.tar.gz...
Extracting big-committee-with-drop-small-degree-20251024-095503.tar.gz...
Extracting big-committee-with-